<a href="https://colab.research.google.com/github/krimits/hotel-review-nlp/blob/main/bilstm-threshold-tuning/hotel-review-nlp/notebooks/04_bilstm_threshold_tuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/krimits/hotel-review-nlp/blob/experiment/bilstm-threshold-tuning/hotel-review-nlp/notebooks/04_bilstm_threshold_tuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seed-100 BiLSTM — dev-only threshold tuning

## Goal

Run one controlled follow-up to the final **unweighted** seed-100 BiLSTM. Training remains unchanged. The only experimental change is that the positive-class decision threshold is selected on the development set using macro-F1, and the frozen test set is evaluated once afterward.

This notebook imports the implementation from the repository; it does not maintain a second copied training loop.

## Setup

Use a GPU runtime (`Runtime → Change runtime type → T4 GPU`). The cell below clones the experiment branch into a separate directory and installs the package plus W&B tracking.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/krimits/hotel-review-nlp.git"
BRANCH = "experiment/bilstm-threshold-tuning"
CHECKOUT_DIR = Path("/content/hotel-review-threshold")

if not CHECKOUT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(CHECKOUT_DIR)],
        check=True,
    )

PROJECT_DIR = CHECKOUT_DIR / "hotel-review-nlp"
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[tracking]"], check=True)

print("Project:", PROJECT_DIR)
print("Branch:", BRANCH)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

Project: /content/hotel-review-threshold/hotel-review-nlp
Branch: experiment/bilstm-threshold-tuning


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

### Connect W&B

Authenticate with your own W&B account. Do not paste an API key into a notebook cell that will be saved to GitHub.

In [2]:
import wandb

wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krimits (krimits-aueb-students-investment-finance-club) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Data

### Key assumptions

- `train.parquet`, `dev.parquet`, and `test.parquet` are the exact frozen splits used by the earlier seed-100 run.
- No split is regenerated with seed 100; changing the split would invalidate the controlled comparison.
- If the previous Colab checkout still exists, the next cell copies the splits from it. Otherwise it opens an upload dialog for the three parquet files.

In [3]:
from pathlib import Path
import shutil

required_files = ("train.parquet", "dev.parquet", "test.parquet")
destination = PROJECT_DIR / "data" / "processed"
destination.mkdir(parents=True, exist_ok=True)

candidate_directories = [
    Path("/content/hotel-review-nlp/hotel-review-nlp/data/processed"),
    Path("/content/data/processed"),
]

if not all((destination / name).exists() for name in required_files):
    for candidate in candidate_directories:
        if all((candidate / name).exists() for name in required_files):
            for name in required_files:
                shutil.copy2(candidate / name, destination / name)
            print("Copied frozen splits from:", candidate)
            break

missing = [name for name in required_files if not (destination / name).exists()]
if missing:
    from google.colab import files

    print("Upload the exact frozen split files:", missing)
    uploaded = files.upload()
    for uploaded_name, content in uploaded.items():
        basename = Path(uploaded_name).name
        if basename in required_files:
            (destination / basename).write_bytes(content)

missing = [name for name in required_files if not (destination / name).exists()]
if missing:
    raise FileNotFoundError(f"Still missing frozen split files: {missing}")

print("All frozen splits are available.")

All frozen splits are available.


### Validate the frozen splits

In [4]:
import pandas as pd

split_summary = []
for split_name in ("train", "dev", "test"):
    frame = pd.read_parquet(destination / f"{split_name}.parquet")
    counts = frame["label"].value_counts().to_dict()
    split_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "negative": int(counts.get("negative", 0)),
            "positive": int(counts.get("positive", 0)),
        }
    )

split_summary = pd.DataFrame(split_summary)
display(split_summary)

expected_rows = {"train": 118_990, "dev": 14_872, "test": 13_278}
observed_rows = dict(zip(split_summary["split"], split_summary["rows"], strict=True))
assert observed_rows == expected_rows, (observed_rows, expected_rows)

,split,rows,negative,positive
0,train,118990,26232,92758
1,dev,14872,3278,11594
2,test,13278,3278,10000


## Run the controlled experiment

The training objective remains ordinary, unweighted cross-entropy. At each epoch, threshold selection uses only the dev set. The checkpoint and threshold with the best dev macro-F1 are then applied once to the test set.

In [11]:
import sys
from pathlib import Path

# Based on the directory inspection, the package is in PROJECT_DIR / 'src'
src_path = str(PROJECT_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from reviewnlp.baselines.bilstm import train_bilstm

print("Import successful!")
final_test_metrics = train_bilstm("configs/bilstm_seed100_threshold.yaml")
final_test_metrics

Import successful!


vocab=15,138 train=118,990 dev=14,872 test=13,278 device=cuda
epoch 01 | loss 0.3087 | dev macro-F1@0.5 0.9410 | dev macro-F1@0.470 0.9418 | 22.7s
epoch 02 | loss 0.1177 | dev macro-F1@0.5 0.9461 | dev macro-F1@0.495 0.9463 | 22.3s
epoch 03 | loss 0.1003 | dev macro-F1@0.5 0.9480 | dev macro-F1@0.475 0.9484 | 21.8s
epoch 04 | loss 0.0875 | dev macro-F1@0.5 0.9475 | dev macro-F1@0.420 0.9483 | 22.4s
epoch 05 | loss 0.0754 | dev macro-F1@0.5 0.9481 | dev macro-F1@0.580 0.9489 | 22.5s
epoch 06 | loss 0.0675 | dev macro-F1@0.5 0.9474 | dev macro-F1@0.685 0.9490 | 21.7s
TEST  macro-F1@0.5=0.9502 | macro-F1@0.685=0.9510 | acc=0.9629


dev/default/accuracy,▁▆█▇█▇
dev/default/macro_f1,▁▆█▇█▇
dev/selected/accuracy,▁▅████
dev/selected/macro_f1,▁▅▇▇██
dev/selected/threshold,▂▃▂▁▅█
epoch,▁▂▄▅▇█
train/loss,█▂▂▂▁▁
best_dev_macro_f1,0.949
dev/default/accuracy,0.9636
dev/default/macro_f1,0.9474
dev/default/macro_precision,0.9448


{'accuracy': 0.9629,
 'macro_f1': 0.951,
 'macro_precision': 0.9442,
 'macro_recall': 0.9583,
 'weighted_f1': 0.9632,
 'per_class': {'negative': {'precision': 0.9054,
   'recall': 0.9491,
   'f1': 0.9267,
   'support': 3278},
  'positive': {'precision': 0.983,
   'recall': 0.9675,
   'f1': 0.9752,
   'support': 10000}},
 'confusion_matrix': [[3111, 167], [325, 9675]]}

## Results

In [10]:
import json

metrics_path = PROJECT_DIR / "runs" / "bilstm_seed100_threshold" / "metrics.json"
with metrics_path.open(encoding="utf-8") as file:
    results = json.load(file)

comparison = pd.DataFrame(
    [
        {"decision_rule": "threshold = 0.5", **results["test_threshold_0_5"]},
        {
            "decision_rule": f"dev-selected threshold = {results['selected_threshold']:.3f}",
            **results["test"],
        },
    ]
)[["decision_rule", "accuracy", "macro_f1", "macro_precision", "macro_recall", "weighted_f1"]]

display(comparison)
print("Selected only on dev:", results["selected_threshold"])
print("Default confusion matrix:", results["test_threshold_0_5"]["confusion_matrix"])
print("Selected-threshold confusion matrix:", results["test"]["confusion_matrix"])

,decision_rule,accuracy,macro_f1,macro_precision,macro_recall,weighted_f1
0,threshold = 0.5,0.9629,0.9502,0.9491,0.9513,0.9629
1,dev-selected threshold = 0.685,0.9629,0.9510,0.9442,0.9583,0.9632


Selected only on dev: 0.685
Default confusion matrix: [[3043, 235], [258, 9742]]
Selected-threshold confusion matrix: [[3111, 167], [325, 9675]]


## Checks

In [12]:
import numpy as np

run_directory = metrics_path.parent
for artifact_name in (
    "dev_logits.npy",
    "dev_labels.npy",
    "test_logits.npy",
    "test_labels.npy",
    "metrics.json",
):
    assert (run_directory / artifact_name).exists(), artifact_name

dev_labels = np.load(run_directory / "dev_labels.npy")
test_labels = np.load(run_directory / "test_labels.npy")
assert len(dev_labels) == 14_872
assert len(test_labels) == 13_278
assert results["selection_metric"] == "dev_macro_f1"

print("All result artifacts and split sizes are valid.")

All result artifacts and split sizes are valid.


## Next Steps

In [13]:
default_macro_f1 = results["test_threshold_0_5"]["macro_f1"]
tuned_macro_f1 = results["test"]["macro_f1"]
delta_pp = 100 * (tuned_macro_f1 - default_macro_f1)

if delta_pp > 0:
    decision = "Retain the dev-selected threshold."
elif delta_pp < 0:
    decision = "Retain the default threshold of 0.5."
else:
    decision = "No measurable difference; retain 0.5 for simplicity."

print(f"Test macro-F1 change: {delta_pp:+.2f} percentage points")
print(decision)
print("Use these executed values to update the PR and stakeholder summary.")

Test macro-F1 change: +0.08 percentage points
Retain the dev-selected threshold.
Use these executed values to update the PR and stakeholder summary.
